In [12]:
import scanpy as sc
import pandas as pd
import pyranges as pr
from pathlib import Path
from scipy import sparse

In [13]:
base_dir = Path.home() / "Library/CloudStorage/OneDrive-WageningenUniversity&Research/InternshipTBB/Data"
gtf_path = base_dir / "references/gencode.vM39.annotation.gtf.gz"
output_dir = base_dir / "output"

# --- LOADING DATA ---
adata_H3K4me3 = sc.read_h5ad(
    f"{base_dir}/gast_H3K4me3_2000_bins_ctype.h5ad"
)

In [14]:
# --- EXPLORING ANNDATA OBJECT FOR H3K4me3 ---
print("object overview:\n")
print(adata_H3K4me3)

print("\nMatrix information:")
print("Shape:", adata_H3K4me3.X.shape)
print("Storage type:", type(adata_H3K4me3.X))
print("Sparse matrix:", sparse.issparse(adata_H3K4me3.X))
print("Data type:", adata_H3K4me3.X.dtype)

print("\nObservation metadata:")
print(adata_H3K4me3.obs.head())

print("\nVariable metadata:")
print(adata_H3K4me3.var.head())

print("\nFirst 5 observation names:")
print(adata_H3K4me3.obs_names[:5])

print("\nFirst 5 variable names:")
print(adata_H3K4me3.var_names[:5])


object overview:

AnnData object with n_obs × n_vars = 18279 × 1364141
    obs: 'sample', 'barcode', 'mark', 'day', 'celltype', 'celltype_general', 'lineage'
    var: 'chrom', 'start', 'end', 'name'

Matrix information:
Shape: (18279, 1364141)
Storage type: <class 'scipy.sparse._csr.csr_matrix'>
Sparse matrix: True
Data type: float32

Observation metadata:
                                                                        sample  \
index                                                                            
PZ-MB-TChIC-Gastd3-rep3-H3K4me3-1::ACACATAG  PZ-MB-TChIC-Gastd3-rep3-H3K4me3-1   
PZ-MB-TChIC-Gastd3-rep3-H3K4me3-1::ACACGAGA  PZ-MB-TChIC-Gastd3-rep3-H3K4me3-1   
PZ-MB-TChIC-Gastd3-rep3-H3K4me3-1::ACACTATC  PZ-MB-TChIC-Gastd3-rep3-H3K4me3-1   
PZ-MB-TChIC-Gastd3-rep3-H3K4me3-1::ACACTGAT  PZ-MB-TChIC-Gastd3-rep3-H3K4me3-1   
PZ-MB-TChIC-Gastd3-rep3-H3K4me3-1::ACAGTACG  PZ-MB-TChIC-Gastd3-rep3-H3K4me3-1   

                                              barcode     mark    

In [15]:
# Convert adata.var into a dataframe with explicit columns
var_H3K4me3_df = adata_H3K4me3.var.copy()

In [16]:
# --- SAVE DATAFRAME AS CSV (already successfully created once)---
var_H3K4me3_df.to_csv(f"{output_dir}/var_H3K4me3.csv",
    index=True,
    index_label="var_index"
)
print("var_df.head:\n", var_H3K4me3_df.head())

var_df.head:
                       chrom  start    end             name
index                                                     
chr1_0_2000::None      chr1      0   2000      chr1_0_2000
chr1_2000_4000::None   chr1   2000   4000   chr1_2000_4000
chr1_4000_6000::None   chr1   4000   6000   chr1_4000_6000
chr1_6000_8000::None   chr1   6000   8000   chr1_6000_8000
chr1_8000_10000::None  chr1   8000  10000  chr1_8000_10000


In [17]:
var_H3K4me3_df.index.name = "var_index"
var_H3K4me3_df = var_H3K4me3_df.reset_index()
# Standardize coordinate column names for PyRanges
intervals_H3K4me3_df = var_H3K4me3_df.rename(columns={
    "chrom": "Chromosome",
    "start": "Start",
    "end": "End"
})
intervals_H3K4me3 = pr.PyRanges(intervals_H3K4me3_df)

In [18]:
# --- LOAD MOUSE REFERENCE GENOME ---
# Load the GTF annotation
gtf_columns = [
    "Chromosome", "Feature", "Start", "End",
    "Score", "Strand", "Attribute"
]

# Reading reference genome genes in chunks:
gene_chunks = []

for chunk in pd.read_csv(
    gtf_path,
    sep="\t",
    comment="#",
    header=None,
    names=[
        "Chromosome", "Source", "Feature", "Start", "End",
        "Score", "Strand", "Frame", "Attribute"
    ],
    usecols=[0, 2, 3, 4, 6, 8],
    compression="infer",
    chunksize=50000
):
    # Discard transcripts and exons immediately
    chunk = chunk.loc[chunk["Feature"] == "gene"].copy()

    if chunk.empty:
        continue

    # Extract gene identifiers and names from GTF attributes
    chunk["gene_id"] = chunk["Attribute"].str.extract(
        r'gene_id "([^"]+)"'
    )
    chunk["gene_name"] = chunk["Attribute"].str.extract(
        r'gene_name "([^"]+)"'
    )

    # GTF uses 1-based inclusive coordinates.
    # Convert Start to PyRanges' 0-based convention.
    chunk["Start"] = chunk["Start"] - 1

    gene_chunks.append(
        chunk[[
            "Chromosome", "Start", "End", "Strand",
            "gene_id", "gene_name"
        ]]
    )


In [19]:
# Combine only the gene records
genes_df = pd.concat(gene_chunks, ignore_index=True)
genes = pr.PyRanges(genes_df)

print("Annotated genes:", len(genes))                               ### 78289
print(genes.df.head())

print("Input intervals H3K4me3:", len(intervals_H3K4me3))           ### 1364141
print(intervals_H3K4me3.df.head())

print(genes.df.head())


Annotated genes: 78289
  Chromosome    Start      End Strand               gene_id      gene_name
0       chr1  3143475  3144545      +  ENSMUSG00000102693.2  4933401J01Rik
1       chr1  3172238  3172348      +  ENSMUSG00000064842.3        Gm26206
2       chr1  3322979  3323459      +  ENSMUSG00000102851.2        Gm18956
3       chr1  3536793  3583776      +  ENSMUSG00000089699.3         Gm1992
4       chr1  3602017  3602943      +  ENSMUSG00000103147.2         Gm7341
Input intervals H3K4me3: 1364141
                     var_index  Chromosome  Start    End  \
0      GL456210.1_0_2000::None  GL456210.1      0   2000   
1   GL456210.1_2000_4000::None  GL456210.1   2000   4000   
2   GL456210.1_4000_6000::None  GL456210.1   4000   6000   
3   GL456210.1_6000_8000::None  GL456210.1   6000   8000   
4  GL456210.1_8000_10000::None  GL456210.1   8000  10000   

                    name  
0      GL456210.1_0_2000  
1   GL456210.1_2000_4000  
2   GL456210.1_4000_6000  
3   GL456210.1_6000_8000 

In [20]:
# --- FIND OVERLAPPING GENES ---
overlaps_H3K4me3 = intervals_H3K4me3.join(genes)
overlaps_H3K4me3_df = overlaps_H3K4me3.df

print("Number of interval-gene overlap pairs (H3K4me3):", len(overlaps_H3K4me3_df))     ### 938147
print(overlaps_H3K4me3_df.head())

overlaps_H3K4me3_df.to_csv(
    f"{output_dir}/interval_gene_overlaps_mm39_H3K4me3.csv",
    index=False
)

join: Strand data from other will be added as strand data to self.
If this is undesired use the flag apply_strand_suffix=False.
To turn off the warning set apply_strand_suffix to True or False.


Number of interval-gene overlap pairs (H3K4me3): 938147
                    var_index Chromosome    Start      End  \
0  chr1_3142000_3144000::None       chr1  3142000  3144000   
1  chr1_3144000_3146000::None       chr1  3144000  3146000   
2  chr1_3172000_3174000::None       chr1  3172000  3174000   
3  chr1_3322000_3324000::None       chr1  3322000  3324000   
4  chr1_3536000_3538000::None       chr1  3536000  3538000   

                   name  Start_b    End_b Strand               gene_id  \
0  chr1_3142000_3144000  3143475  3144545      +  ENSMUSG00000102693.2   
1  chr1_3144000_3146000  3143475  3144545      +  ENSMUSG00000102693.2   
2  chr1_3172000_3174000  3172238  3172348      +  ENSMUSG00000064842.3   
3  chr1_3322000_3324000  3322979  3323459      +  ENSMUSG00000102851.2   
4  chr1_3536000_3538000  3536793  3583776      +  ENSMUSG00000089699.3   

       gene_name  
0  4933401J01Rik  
1  4933401J01Rik  
2        Gm26206  
3        Gm18956  
4         Gm1992  


In [21]:
# --- Count how many genes overlap each interval ---
interval_gene_counts = intervals_H3K4me3.count_overlaps(genes).df
interval_gene_counts.to_csv(
    f"{output_dir}/interval_gene_counts_mm39_H3K4me3.csv",
    index=False
)
